In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

for folder in sorted(drive_root.iterdir()):
    if folder.is_dir():
        print(folder.name)

Colab Notebooks
Distributional-Adversarial-Attacks-and-Training-in-Deep-Hedging-main
Fiver
Instagram
London-Skopje


In [3]:
from pathlib import Path
import sys
import json
import hashlib
import time

import numpy as np
import pandas as pd
import torch
import cvxpy as cp
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Adjust this to the location of your project in Google Drive.
root = Path("/content/drive/MyDrive/awdro")

if not root.is_dir():
    raise FileNotFoundError(f"Project folder not found: {root}")

for folder in [
    root,
    root / "hedging",
    root / "path simulation",
    root / "inner maximization",
    root / "outer minimization",
]:
    sys.path.insert(0, str(folder))

from Heston import Heston
from SuperhedgingProblem import SuperhedgingProblem
from NonrectangularSolver import NonrectangularSolver
from RectangularSolver import RectangularSolver
from RelaxedSuperhedgingSolver import RelaxedSuperhedgingSolver

if "HIGHS" not in cp.installed_solvers():
    raise RuntimeError("HiGHS is unavailable. Restart after installing highspy.")

# First run: check runtime with 3 seeds.
# Then change to 50 or 100 and rerun.
N_SEEDS = 3
SEEDS = range(1000, 1000 + N_SEEDS)

N_STEPS_LIST = [2, 3, 4]
BRANCHES = 2

MATURITY = 1.0
BUDGET = 1.0
PENALTY = 10.0
STRIKE = 100.0

CAPITAL_BOUNDS = (0.0, 50.0)
HOLDING_BOUNDS = (-2.0, 2.0)

ABS_TOL = 1e-5
REL_TOL = 1e-5

# Change this tag after modifying your solver implementation.
CODE_VERSION = "finite_model_v1"

MODEL_PARAMETERS = {
    "s0": 100.0,
    "v0": 0.04,
    "mu": 0.0,
    "kappa": 2.0,
    "theta": 0.04,
    "vol_of_vol": 0.3,
    "rho": -0.7,
}

config = {
    "code_version": CODE_VERSION,
    "model": MODEL_PARAMETERS,
    "maturity": MATURITY,
    "budget": BUDGET,
    "penalty": PENALTY,
    "strike": STRIKE,
    "branches": BRANCHES,
    "capital_bounds": CAPITAL_BOUNDS,
    "holding_bounds": HOLDING_BOUNDS,
    "payoff": "European call",
    "cost": "sum of squared stock and variance level differences",
    "abs_tol": ABS_TOL,
    "rel_tol": REL_TOL,
}

study_id = hashlib.sha256(
    json.dumps(config, sort_keys=True).encode()
).hexdigest()[:12]

output_dir = root / "results" / f"time_grid_{study_id}"
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "configuration.json", "w") as file:
    json.dump(config, file, indent=2)

print(f"Results: {output_dir}")
print(f"Requested paired experiments: {len(N_STEPS_LIST) * N_SEEDS}")

for steps in N_STEPS_LIST:
    reference_count = BRANCHES**steps
    candidate_count = BRANCHES ** (steps * (steps + 1) // 2)
    print(
        f"{steps} intervals: up to "
        f"{reference_count:,} reference paths, "
        f"{candidate_count:,} adversarial paths, "
        f"{reference_count * candidate_count:,} coupling variables"
    )

FileNotFoundError: Project folder not found: /content/drive/MyDrive/awdro

In [ ]:
def call_payoff(paths):
    return torch.clamp(paths[:, -1, 0] - STRIKE, min=0.0)


def run_experiment(seed, n_steps):
    started = time.perf_counter()

    row = {
        "seed": int(seed),
        "n_steps": int(n_steps),
        "status": "failed",
        "stage": "construction",
        "error": "",
    }

    try:
        model = Heston(**MODEL_PARAMETERS)
        times = np.linspace(0.0, MATURITY, n_steps + 1)

        problem = SuperhedgingProblem(
            payoff=call_payoff,
            trading_times=times,
        )

        discretization = {
            "seed": int(seed),
            "reference_branches": BRANCHES,
        }

        rectangular = RectangularSolver(
            model, times, BUDGET, discretization
        )
        nonrectangular = NonrectangularSolver(
            model, times, BUDGET, discretization
        )

        row["stage"] = "shared_model_check"

        np.testing.assert_array_equal(
            np.asarray([
                node["state"] for node in rectangular.reference_nodes
            ]),
            np.asarray([
                node["state"] for node in nonrectangular.reference_nodes
            ]),
        )

        for t in range(n_steps + 1):
            np.testing.assert_array_equal(
                rectangular.adversarial_candidates[t],
                nonrectangular.adversarial_candidates[t],
            )

        outer = RelaxedSuperhedgingSolver(
            problem=problem,
            inner_solver=nonrectangular,
            penalty=PENALTY,
        )

        methods = {
            "nonrectangular": outer.solve_adaptive_hedge_nonrectangular,
            "rectangular": outer.solve_adaptive_hedge_rectangular,
        }

        for name, method in methods.items():
            row["stage"] = f"{name}_optimization"
            optimization_started = time.perf_counter()

            result = method(
                rectangular,
                CAPITAL_BOUNDS,
                HOLDING_BOUNDS,
            )

            row[f"{name}_seconds"] = (
                time.perf_counter() - optimization_started
            )

            value = float(result["value"])
            capital = float(result["capital"])
            shortfall = float(result["worst_case_shortfall"])
            holdings = np.asarray(result["holdings"], dtype=float)

            if not np.isfinite([value, capital, shortfall]).all():
                raise RuntimeError(f"{name}: nonfinite result.")

            if holdings.size == 0 or not np.isfinite(holdings).all():
                raise RuntimeError(f"{name}: invalid holdings.")

            np.testing.assert_allclose(
                value,
                capital + PENALTY * shortfall,
                atol=ABS_TOL,
                rtol=REL_TOL,
            )

            capital_tol = ABS_TOL + REL_TOL * max(
                1.0, *map(abs, CAPITAL_BOUNDS)
            )
            holding_tol = ABS_TOL + REL_TOL * max(
                1.0, *map(abs, HOLDING_BOUNDS)
            )
            value_tol = ABS_TOL + REL_TOL * max(1.0, abs(value))

            if shortfall < -value_tol:
                raise RuntimeError(f"{name}: negative shortfall.")

            if not (
                CAPITAL_BOUNDS[0] - capital_tol
                <= capital
                <= CAPITAL_BOUNDS[1] + capital_tol
            ):
                raise RuntimeError(f"{name}: capital outside bounds.")

            if (
                (holdings < HOLDING_BOUNDS[0] - holding_tol).any()
                or (holdings > HOLDING_BOUNDS[1] + holding_tol).any()
            ):
                raise RuntimeError(f"{name}: holdings outside bounds.")

            row[f"{name}_value"] = value
            row[f"{name}_capital"] = capital
            row[f"{name}_shortfall"] = shortfall

            row[f"{name}_capital_at_bound"] = bool(
                np.isclose(
                    capital, CAPITAL_BOUNDS,
                    atol=capital_tol, rtol=0.0,
                ).any()
            )

            row[f"{name}_holding_bound_fraction"] = float(
                (
                    np.isclose(
                        holdings, HOLDING_BOUNDS[0],
                        atol=holding_tol, rtol=0.0,
                    )
                    | np.isclose(
                        holdings, HOLDING_BOUNDS[1],
                        atol=holding_tol, rtol=0.0,
                    )
                ).mean()
            )

        row["stage"] = "gap_check"

        gap = row["rectangular_value"] - row["nonrectangular_value"]
        tolerance = ABS_TOL + REL_TOL * max(
            1.0,
            abs(row["rectangular_value"]),
            abs(row["nonrectangular_value"]),
        )

        row["gap"] = gap
        row["positive_above_tolerance"] = bool(gap > tolerance)

        if gap < -tolerance:
            raise RuntimeError(
                f"Negative gap {gap:.6g} exceeds tolerance {tolerance:.6g}."
            )

        if BUDGET == 0 and abs(gap) > tolerance:
            raise RuntimeError("Zero-budget objectives disagree.")

        row["status"] = "ok"
        row["stage"] = "complete"

    except Exception as exc:
        row["error"] = f"{type(exc).__name__}: {exc}"

    row["total_seconds"] = time.perf_counter() - started
    return row